# experiment_wdi_resource_curse

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment_wdi_resource_curse"
DATA_FILENAME = "wdi_reversal_panel.csv"   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


# WDI resource-curse edge: resource_rents -> gdp_growth

**Dataset.** World Development Indicators panel, 265 units x up to 54 years (1970--2023), 9 development indicators. Diagnostic-screened as RICH support: the growth-driver subset {gdp_growth, resource_rents, investment, trade_openness, fdi} has pre-fit source r_eff 4.72/5 (levels), matching/exceeding Beijing. This is the first real candidate for a SECOND clean-recovery domain.

**Edge (pre-committed).** Target `gdp_growth` (a rate; levels). Edge source `resource_rents`; candidate modulators `investment`, `trade_openness`, `fdi`. Substantive hypothesis: the CONDITIONAL RESOURCE CURSE -- the effect of resource rents on growth is modulated by how rents are channeled into productive capital.

**Pre-committed prediction.** Among {investment, trade_openness, fdi}, `investment` ranks first as modulator of the resource_rents->gdp_growth edge, seed-stable. Named theory-competitor: `trade_openness` (Sachs-Warner openness strand) -- if it wins instead, that is the data favoring one resource-curse mechanism, reported as such, not a failure.

**Outcomes (all reportable).** (1) seed-stable dominant modulator = clean second positive; (2) seed-stable but near-trivial gates (G-NAVAR ~ additive) = rich support but weak interaction structure (diagnostic certifies IDENTIFIABILITY, not EXISTENCE of an effect); (3) seed-unstable despite rich support = surprising tension. We report whichever occurs. Outputs: results.csv, metadata.json, story.txt.

## Cell 1: Drive + data

## Cell 2: Config

In [ ]:
TARGET='gdp_growth'
EDGE='resource_rents'
SOURCES=['resource_rents','investment','trade_openness','fdi']  # edge source first
PREDICTED='investment'; COMPETITOR='trade_openness'
K=2; BASE_SEED=42; N_RESTARTS=3; N_STABILITY_SEEDS=2; TEST_FRACTION=0.20
def make_cfg(): return Config(n_vars=1+len(SOURCES),K=K,hidden_dim=32,n_epochs=300,l1_lambda=0.005,triviality_threshold=0.001)
print('edge',EDGE,'->',TARGET,'| modulators:',[s for s in SOURCES if s!=EDGE],'| predicted:',PREDICTED)

## Cell 3: Build (contiguous-year runs, train-only z-scoring)

In [ ]:
df_raw=pd.read_csv(DATA_PATH).sort_values(['ISO3','year'])
def build_cell(df):
    Xtr_l,ytr_l,Xte_l,yte_l=[],[],[],[]
    for iso,g in df.groupby('ISO3'):
        s=g.sort_values('year')[['year',TARGET]+SOURCES].dropna()
        n=len(s)
        if n<2*(K+1): continue
        years=s['year'].values; M=s[[TARGET]+SOURCES].values.astype(np.float64)
        cut=int(n*(1-TEST_FRACTION))
        if cut<K+1 or n-cut<K+1: continue
        mu=M[:cut].mean(0); sd=M[:cut].std(0)+1e-9; Mz=(M-mu)/sd  # TRAIN-ONLY stats
        for (lo,hi),XL,yL in [((0,cut),Xtr_l,ytr_l),((cut,n),Xte_l,yte_l)]:
            runs=[]; seg=lo
            for i in range(lo+1,hi+1):
                if i==hi or years[i]!=years[i-1]+1:
                    if i-seg>=K+1: runs.append([seg,i])
                    seg=i
            if runs:
                Xlag,y=make_lag_tensor_runs(Mz,np.array(runs),K=K,target_col=0)
                if len(y)>0: XL.append(Xlag); yL.append(y)
    Xtr=np.concatenate(Xtr_l); ytr=np.concatenate(ytr_l)
    Xte=np.concatenate(Xte_l); yte=np.concatenate(yte_l)
    assert Xtr.shape[1]==len(SOURCES)
    # PRE-COMMITTED tail treatment: winsorize target + all sources at TRAIN 1st/99th pct (leakage-free),
    # standard for a heavy-tailed growth target; caps from train only, applied to train and test.
    ylo,yhi=np.percentile(ytr,[1,99]); ytr=np.clip(ytr,ylo,yhi); yte=np.clip(yte,ylo,yhi)
    for j in range(Xtr.shape[1]):
        lo,hi=np.percentile(Xtr[:,j,:],[1,99])
        Xtr[:,j,:]=np.clip(Xtr[:,j,:],lo,hi); Xte[:,j,:]=np.clip(Xte[:,j,:],lo,hi)
    # np.clip/np.percentile upcast to float64; model params are float32 -> cast back to avoid dtype crash
    Xtr=Xtr.astype(np.float32); ytr=ytr.astype(np.float32)
    Xte=Xte.astype(np.float32); yte=yte.astype(np.float32)
    return Xtr,ytr,Xte,yte
Xtr,ytr,Xte,yte=build_cell(df_raw)
print(f'n_train={Xtr.shape[0]}, n_test={Xte.shape[0]}, source-support r_eff={effective_rank_full(Xtr):.2f} / {len(SOURCES)}')

## Cell 4: Fit (2 seeds + additive), rank + gate sign

In [ ]:
@torch.no_grad()
def edge_ranking(model,Xt,src,edge):
    j=src.index(edge); sc=[(src[k],gate_triviality_score(model,Xt,j,k)) for k in range(len(src)) if k!=j]
    sc.sort(key=lambda x:-x[1]); return sc
@torch.no_grad()
def gate_slope(model,Xt,src,edge,mod):
    j=src.index(edge); k=src.index(mod); xb=Xt[:,k,:]
    g=model.evaluate_gate(j=j,k=k,x_block=xb).cpu().numpy().reshape(-1); m=float(np.mean(g)); g=g/m if m>1e-12 else g
    xk=xb.mean(1).cpu().numpy(); lo=xk<np.quantile(xk,0.33); hi=xk>np.quantile(xk,0.67)
    return float(g[hi].mean()-g[lo].mean()) if (lo.sum()>=5 and hi.sum()>=5) else float('nan')
r_eff=effective_rank_full(Xtr); cfg=make_cfg(); Xt=torch.from_numpy(Xtr).float().to(DEVICE); seeds=[]
for sdi in range(N_STABILITY_SEEDS):
    m=fit_gnavar_from_lag_with_restarts(Xtr,ytr,cfg,seed=BASE_SEED+1000*sdi,n_restarts=N_RESTARTS,verbose=False)
    rk=edge_ranking(m,Xt,SOURCES,EDGE); mse=held_out_mse_gnavar_from_lag(m,Xte,yte)
    slopes={mod:gate_slope(m,Xt,SOURCES,EDGE,mod) for mod in SOURCES if mod!=EDGE}
    seeds.append({'rk':rk,'top':rk[0][0],'mse':mse,'slopes':slopes})
    print(f'seed{sdi}: top={rk[0][0]}  ranking='+', '.join(f'{n}={s:.4f}' for n,s in rk)+f'  mse={mse:.4f}')
mpw=fit_pairwise_from_lag_with_restarts(Xtr,ytr,cfg,seed=BASE_SEED,n_restarts=N_RESTARTS,verbose=False)
mse_pw=held_out_mse_pairwise_from_lag(mpw,Xte,yte)
print(f'additive MSE={mse_pw:.4f}  | ratio additive/gnavar={mse_pw/seeds[0]["mse"]:.3f}')

## Cell 5: Verdict + save

In [ ]:
agree=seeds[0]['top']==seeds[1]['top']
row={'edge':EDGE,'target':TARGET,'r_eff':r_eff,'n_train':Xtr.shape[0],'n_test':Xte.shape[0],
  'predicted_modulator':PREDICTED,'top_seed0':seeds[0]['top'],'top_seed1':seeds[1]['top'],'seed_agree':agree,
  'predicted_wins':seeds[0]['top']==PREDICTED and agree,
  'mse_gnavar':seeds[0]['mse'],'mse_additive':mse_pw,'mse_ratio_add_to_gn':mse_pw/seeds[0]['mse'],
  'ranking_seed0':json.dumps(seeds[0]['rk']),'ranking_seed1':json.dumps(seeds[1]['rk']),
  'slopes_seed0':json.dumps(seeds[0]['slopes']),'slopes_seed1':json.dumps(seeds[1]['slopes'])}
pd.DataFrame([row]).to_csv(RESULTS_DIR/'results.csv',index=False)
L=['WDI resource-curse: resource_rents -> gdp_growth','='*48,
   f'source-support r_eff={r_eff:.2f}/{len(SOURCES)} (rich; diagnostic predicts recovery feasible)',
   f'n_train={Xtr.shape[0]}, n_test={Xte.shape[0]}','',
   f'PRE-COMMITTED prediction: {PREDICTED} ranks #1, seed-stable. (competitor: {COMPETITOR})','',
   f'seed0 top={seeds[0]["top"]}: '+', '.join(f'{n}={s:.4f}' for n,s in seeds[0]['rk']),
   f'seed1 top={seeds[1]["top"]}: '+', '.join(f'{n}={s:.4f}' for n,s in seeds[1]['rk']),
   f'seed agreement: {agree}',
   f'gate slopes seed0: {seeds[0]["slopes"]}',
   f'gate slopes seed1: {seeds[1]["slopes"]}',
   f'MSE: G-NAVAR={seeds[0]["mse"]:.4f}  additive={mse_pw:.4f}  (ratio add/gn={mse_pw/seeds[0]["mse"]:.3f})','',
   'OUTCOME READING:',
   f'  (1) seed_agree=True & top={PREDICTED} & gates non-trivial -> clean second positive (predicted)',
   f'  (2) seed_agree=True & top={COMPETITOR} -> resource-curse via openness, reported as competitor win',
   '  (3) seed_agree=True but ratio~1.0 & flat gates -> rich support, weak interaction (identifiability != existence)',
   '  (4) seed_agree=False -> unstable despite rich support (surprising tension)']
s='\n'.join(L); (RESULTS_DIR/'story.txt').write_text(s); print(s)
def _sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(8192),b''): h.update(c)
    return h.hexdigest()
import gnavar_core as _gc
json.dump({'timestamp':datetime.datetime.now(datetime.timezone.utc).isoformat(),'experiment':'wdi_resource_curse',
  'data_sha256':_sha(DATA_PATH),'gnavar_core_sha256':_sha(_gc.__file__),
  'target':TARGET,'edge':EDGE,'sources':SOURCES,'predicted_modulator':PREDICTED,'competitor':COMPETITOR,
  'note':'rich-support real domain (r_eff 4.72/5 pre-screen); pre-committed conditional-resource-curse edge; levels'},
  open(RESULTS_DIR/'metadata.json','w'),indent=2)
print('\nsaved')